# Agent IA

**Objectif** : construire un agent qui peut **utiliser des outils** pour répondre à des questions qu'un LLM seul ne saurait pas traiter.

## Un LLM seul ne peut pas…

Essayez de demander à ChatGPT :
- *"Quelle température fait-il à Paris **maintenant** ?"* → il n'a pas accès au temps réel
- *"Combien fait 4852 × 37.6 ?"* → il devine, souvent faux
- *"Quelle est l'actualité d'aujourd'hui ?"* → connaissances figées à sa date d'entraînement

**Un LLM, c'est puissant pour raisonner, mais il ne sait pas agir dans le monde réel.**

## Un agent = LLM + outils + boucle de décision

Un **agent**, c'est un LLM qu'on connecte à des **outils** (fonctions Python, APIs…). Le LLM **décide** lui-même quel outil appeler selon la question.

```
Question utilisateur
       ↓
┌──────────────────────┐
│ LLM : je réfléchis │ ← "Reason"
│ Quel outil utiliser ? │
└──────────────────────┘
       ↓
┌──────────────────────┐
│ Appel d'outil      │ ← "Act"
│ get_weather("Paris")  │
└──────────────────────┘
       ↓
┌──────────────────────┐
│ Résultat observé   │ ← "Observe"
│ "18°C, nuageux"       │
└──────────────────────┘
       ↓
   (peut reboucler)
       ↓
┌──────────────────────┐
│ Réponse finale     │
└──────────────────────┘
```

---
## Étape 0 — Setup

In [1]:
!pip install -q openai requests

In [3]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ Clé API configurée")

✅ Clé API configurée


---
## Étape 1 — Créer les outils (fonctions Python)

Un outil = juste une **fonction Python normale**.

### Outil 1 : `get_weather` — API Open-Meteo (gratuite, sans clé)

In [2]:
import requests

def get_weather(ville: str) -> str:
    """Récupère la météo actuelle d'une ville via Open-Meteo."""

    # Étape 1 : convertir le nom de la ville en coordonnées (géocodage)
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": ville, "count": 1, "language": "fr"}
    ).json()

    if not geo.get("results"):
        return f"Ville '{ville}' introuvable"

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    nom = geo["results"][0]["name"]
    pays = geo["results"][0].get("country", "")

    # Étape 2 : récupérer la météo
    meteo = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"
        }
    ).json()

    temp = meteo["current"]["temperature_2m"]
    vent = meteo["current"]["wind_speed_10m"]
    humidite = meteo["current"]["relative_humidity_2m"]

    return f"Météo à {nom} ({pays}) : {temp}°C, vent {vent} km/h, humidité {humidite}%"


# Test en direct
print(get_weather("Paris"))
print(get_weather("Tokyo"))

Météo à Paris (France) : 13.6°C, vent 11.0 km/h, humidité 46%
Météo à Tokyo (Japon) : 12.8°C, vent 3.4 km/h, humidité 93%


### Outil 2 : `calculator` — évaluer une expression mathématique

In [4]:
def calculator(expression: str) -> str:
    """Évalue une expression mathématique. Ex : '(23 + 17) * 2'"""
    try:
        # eval() normalement dangereux, mais on restreint aux maths de base
        # Pour un vrai agent en prod, utiliser ast.literal_eval ou un parseur dédié
        autorise = "0123456789+-*/(). "
        if not all(c in autorise for c in expression):
            return "Erreur : caractères non autorisés"
        resultat = eval(expression)
        return f"Résultat : {resultat}"
    except Exception as e:
        return f"Erreur de calcul : {e}"


# Test
print(calculator("(23 + 17) * 2"))
print(calculator("125.5 / 3"))

Résultat : 80
Résultat : 41.833333333333336


---
## Étape 2 — Décrire les outils au LLM

Pour que le LLM sache **quels outils existent** et **comment les utiliser**, il faut les lui décrire dans un format standardisé.

C'est comme une **documentation** pour le LLM : il lit la description, les paramètres, et décide quand l'appeler.

In [5]:
tools_description = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Récupère la météo actuelle d\'une ville (température, vent, humidité). À utiliser pour toute question sur le temps qu\'il fait.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ville": {
                        "type": "string",
                        "description": "Le nom de la ville (ex: Paris, New York, Tokyo)"
                    }
                },
                "required": ["ville"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Évalue une expression mathématique. À utiliser pour tout calcul numérique.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "L\'expression à évaluer, ex: '(23 + 17) * 2' ou '125.5 / 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# Dictionnaire pour appeler les fonctions par leur nom
tools_mapping = {
    "get_weather": get_weather,
    "calculator": calculator
}

print(f"✅ {len(tools_description)} outils déclarés")
for t in tools_description:
    print(f"   - {t['function']['name']} : {t['function']['description'][:60]}...")

✅ 2 outils déclarés
   - get_weather : Récupère la météo actuelle d'une ville (température, vent, h...
   - calculator : Évalue une expression mathématique. À utiliser pour tout cal...


In [23]:
# Add the new tool to the tools_description list
tools_description.append(
    {
        "type": "function",
        "function": {
            "name": "get_pokemon_weakness",
            "description": "Détermine les types d'attaques les plus efficaces contre un ou deux types de Pokémon défenseurs. À utiliser pour toute question sur les faiblesses ou résistances de Pokémon.",
            "parameters": {
                "type": "object",
                "properties": {
                    "defending_types_str": {
                        "type": "string",
                        "description": "Un ou deux types de Pokémon défenseurs séparés par un slash (ex: 'Grass', 'Fire/Water')."
                    }
                },
                "required": ["defending_types_str"]
            }
        }
    }
)

# Add the new tool to the tools_mapping dictionary
tools_mapping["get_pokemon_weakness"] = get_pokemon_weakness

print(f"✅ {len(tools_description)} outils déclarés après ajout de get_pokemon_weakness")
for t in tools_description:
    print(f"   - {t['function']['name']} : {t['function']['description'][:60]}...")

✅ 4 outils déclarés après ajout de get_pokemon_weakness
   - get_weather : Récupère la météo actuelle d'une ville (température, vent, h...
   - calculator : Évalue une expression mathématique. À utiliser pour tout cal...
   - get_pokemon_weakness : Détermine les types d'attaques les plus efficaces contre un ...
   - get_pokemon_weakness : Détermine les types d'attaques les plus efficaces contre un ...


### Test du nouvel outil Pokémon

---
 ## Étape 9 — Créer un outil pour trouver le type d'un Pokémon.

In [25]:
import pandas as pd

# Load the Pokémon list CSV file into a DataFrame
pokemon_list_df = pd.read_csv('/content/Platinum Kaizo Docs - Pokemon List.csv')

# Display the first few rows and the columns to understand the structure
print('Pokémon List Chart:')
display(pokemon_list_df.head())
print('\nColumn Information:')
pokemon_list_df.info()

Pokémon List Chart:


,ID Number,Name,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,BST,Type 1,...,Gender Ratio,Hatch Multiplier,Base Happiness,Growth Rate,Egg Group 1,Egg Group 2,Ability 1,Ability 2,Safari Zone Run Chance,DO NOT TOUCH
0,0,-----,0,0,0,0,0,0,0,Normal,...,0,0,0,Medium Fast,~,~,-,-,0,0
1,1,BULBASAUR,45,49,49,65,65,45,318,Grass,...,31,20,70,Medium Slow,Monster,Grass,Overgrow,Overgrow,45,3
2,2,IVYSAUR,60,62,63,80,80,60,405,Grass,...,31,20,70,Medium Slow,Monster,Grass,Overgrow,Overgrow,45,3
3,3,VENUSAUR,80,92 (+10),83,100,100,80,535 (+10),Grass,...,31,20,70,Medium Slow,Monster,Grass,Chlorophyll,Overgrow,45,3
4,4,CHARMANDER,39,52,43,60,50,65,309,Fire,...,31,20,70,Medium Slow,Monster,Dragon,Blaze,Blaze,0,0



Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ID Number               508 non-null    int64 
 1   Name                    508 non-null    object
 2   HP                      508 non-null    object
 3   Attack                  508 non-null    object
 4   Defense                 508 non-null    object
 5   Sp. Atk                 508 non-null    object
 6   Sp. Def                 508 non-null    object
 7   Speed                   508 non-null    object
 8   BST                     508 non-null    object
 9   Type 1                  508 non-null    object
 10  Type 2                  508 non-null    object
 11  Catch Rate              508 non-null    int64 
 12  Exp Drop                508 non-null    int64 
 13  Uncommon Held Item      12 non-null     object
 14  Rare Held Item          12 non-null  

In [31]:
def get_pokemon_typing(pokemon_name: str) -> str:
    """Récupère le ou les types d'un Pokémon donné à partir de la liste des Pokémon.
    À utiliser pour trouver le type d'un Pokémon spécifique."""
    pokemon_name_lower = pokemon_name.lower()

    # Try to find the Pokémon by name
    result = pokemon_list_df[pokemon_list_df['Name'].str.lower() == pokemon_name_lower]

    if result.empty:
        return f"Pokémon '{pokemon_name}' introuvable dans la liste."

    # Assuming 'Type 1' and 'Type 2' columns exist
    type1 = result['Type 1'].iloc[0]
    type2 = result['Type 2'].iloc[0]

    if pd.isna(type2) or type2 == '':
        return f"Le Pokémon {pokemon_name} est de type {type1}."
    else:
        return f"Le Pokémon {pokemon_name} est de type {type1}/{type2}."

# Test the function directly
print(get_pokemon_typing('Charizard'))
print(get_pokemon_typing('Pikachu'))
print(get_pokemon_typing('Arceus'))
print(get_pokemon_typing('Missingno'))

Le Pokémon Charizard est de type Fire/Flying.
Le Pokémon Pikachu est de type Electric/Electric.
Le Pokémon Arceus est de type Normal/Normal.
Pokémon 'Missingno' introuvable dans la liste.


In [32]:
# Add the new tool to the tools_description list
tools_description.append(
    {
        "type": "function",
        "function": {
            "name": "get_pokemon_typing",
            "description": "Récupère le ou les types d'un Pokémon donné. À utiliser pour toute question sur le type d'un Pokémon.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_name": {
                        "type": "string",
                        "description": "Le nom du Pokémon (ex: Pikachu, Charizard)"
                    }
                },
                "required": ["pokemon_name"]
            }
        }
    }
)

# Add the new tool to the tools_mapping dictionary
tools_mapping["get_pokemon_typing"] = get_pokemon_typing

print(f"✅ {len(tools_description)} outils déclarés après ajout de get_pokemon_typing")
for t in tools_description:
    print(f"   - {t['function']['name']} : {t['function']['description'][:60]}...")

✅ 5 outils déclarés après ajout de get_pokemon_typing
   - get_weather : Récupère la météo actuelle d'une ville (température, vent, h...
   - calculator : Évalue une expression mathématique. À utiliser pour tout cal...
   - get_pokemon_weakness : Détermine les types d'attaques les plus efficaces contre un ...
   - get_pokemon_weakness : Détermine les types d'attaques les plus efficaces contre un ...
   - get_pokemon_typing : Récupère le ou les types d'un Pokémon donné. À utiliser pour...


### Test du nouvel outil Pokémon Typing

In [ ]:
# Test pour un Pokémon à un seul type
qagent("Quel est le type de Pikachu ?"))

# Test pour un Pokémon à deux types
print(agent("Quel est le type de Charizard ?"))

# Test pour un Pokémon qui n'est pas dans la liste
print(agent("Quel est le type de Arceus ?"))

# Test pour un Pokémon avec un nom invalide/inexistant
print(agent("Quel est le type de Missingno ?"))

In [24]:
# Test pour un seul type
print(agent("Quel est le type d'attaque le plus efficace contre un Pokémon de type Feu ?"))

# Test pour une combinaison de deux types
print(agent("Contre un Pokémon de type Herbe/Sol, quels sont les types d'attaques les plus efficaces ?"))

# Test pour un type résistant
print(agent("Quel est le type d'attaque le plus efficace contre un Pokémon de type Normal ?"))

# Test pour un type invalide
print(agent("Quel est le type d'attaque le plus efficace contre un Pokémon de type Inconnu ?"))

❓ Question : Quel est le type d'attaque le plus efficace contre un Pokémon de type Feu ?

 Le LLM veut appeler : get_pokemon_weakness({'defending_types_str': 'Fire'})
 Résultat : Les types d'attaque les plus efficaces contre Fire sont Water, Ground, Rock avec un multiplicateur de 2.0x.

 Réponse finale : Les types d'attaque les plus efficaces contre un Pokémon de type Feu sont :

- Eau (Water)
- Sol (Ground)
- Roche (Rock)

Ces types infligent un multiplicateur de 2.0x de dégâts.
Les types d'attaque les plus efficaces contre un Pokémon de type Feu sont :

- Eau (Water)
- Sol (Ground)
- Roche (Rock)

Ces types infligent un multiplicateur de 2.0x de dégâts.
❓ Question : Contre un Pokémon de type Herbe/Sol, quels sont les types d'attaques les plus efficaces ?

 Le LLM veut appeler : get_pokemon_weakness({'defending_types_str': 'Grass/Ground'})
 Résultat : Le type d'attaque le plus efficace contre Grass/Ground est Ice avec un multiplicateur de 4.0x.

 Réponse finale : Les types d'attaques 

**Point-clé** : c'est **le champ `description`** qui permet au LLM de savoir **quand** utiliser l'outil. Plus la description est claire, mieux l'agent fonctionne.

*Mauvaise description* : `"fonction météo"` → trop vague  
*Bonne description* : `"Récupère la météo actuelle d'une ville (température, vent, humidité). À utiliser pour toute question sur le temps qu'il fait."` → le LLM sait exactement quand s'en servir.

---
## Étape 3 — Premier appel : voir le LLM **décider** d'utiliser un outil

On va envoyer une question au LLM **avec** la liste des outils disponibles. Deux choses peuvent arriver :
1. Le LLM répond directement (pas besoin d'outil)
2. Le LLM renvoie une **demande d'appel d'outil** (`tool_call`)

In [6]:
from openai import OpenAI

client = OpenAI()

question = "Quelle température fait-il à Paris ?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}],
    tools=tools_description  # ← on lui donne les outils !
)

message = response.choices[0].message

print(f"❓ Question : {question}\n")
print(f"📨 Réponse brute du LLM :")
print(f"   content : {message.content}")
print(f"   tool_calls : {message.tool_calls}")

❓ Question : Quelle température fait-il à Paris ?

📨 Réponse brute du LLM :
   content : None
   tool_calls : [ChatCompletionMessageFunctionToolCall(id='call_vJLRYfo955VDo0LRJRm0CuiJ', function=Function(arguments='{"ville":"Paris"}', name='get_weather'), type='function')]


**Observez** : `content` est `None`, mais `tool_calls` contient une demande d'appel à `get_weather` avec `{"ville": "Paris"}`.

**Le LLM n'a pas répondu, il a demandé à appeler un outil.** C'est nous (notre code) qui devons **exécuter** l'outil et lui renvoyer le résultat.

C'est ça, le pattern agent : le LLM **décide**, notre code **exécute**.

---
## Étape 4 — Construire la boucle complète

Le pattern complet :
1. Envoyer la question au LLM
2. Si le LLM demande un outil → l'exécuter et lui renvoyer le résultat
3. Répéter jusqu'à ce qu'il donne une réponse finale

On va faire tout ça avec des **logs visuels** pour voir ce qui se passe.

In [17]:
import json

def agent(question, verbose=True):
    """Agent complet avec boucle ReAct."""

    messages = [{"role": "user", "content": question}]

    if verbose:
        print(f"❓ Question : {question}\n")

    # Boucle : le LLM peut enchaîner plusieurs appels d'outils
    for iteration in range(5):  # max 5 itérations pour éviter les boucles infinies

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_description
        )

        message = response.choices[0].message
        messages.append(message)  # on garde l'historique complet

        # Cas 1 : le LLM donne une réponse finale
        if not message.tool_calls:
            if verbose:
                print(f" Réponse finale : {message.content}")
            return message.content

        # Cas 2 : le LLM demande d'appeler un ou plusieurs outils
        for tool_call in message.tool_calls:
            nom_outil = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if verbose:
                print(f" Le LLM veut appeler : {nom_outil}({arguments})")

            # Exécuter l'outil
            fonction = tools_mapping[nom_outil]
            resultat = fonction(**arguments)

            if verbose:
                print(f" Résultat : {resultat}\n")

            # Renvoyer le résultat au LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": resultat
            })

    return "Trop d\'itérations, abandon."


# Premier test
test1 = agent("Quelle température fait-il à Paris ?")

❓ Question : Quelle température fait-il à Paris ?

 Le LLM veut appeler : get_weather({'ville': 'Paris'})
 Résultat : Météo à Paris (France) : 14.2°C, vent 11.8 km/h, humidité 45%

 Réponse finale : La température à Paris est de 14,2°C, avec un vent de 11,8 km/h et une humidité de 45%.


In [15]:
message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_vJLRYfo955VDo0LRJRm0CuiJ', function=Function(arguments='{"ville":"Paris"}', name='get_weather'), type='function')])

---
## Étape 5 — Parfois, l'agent n'utilise pas d'outil

Si la question peut se répondre sans outil, l'agent répond directement.

In [16]:
agent("Qui a écrit Les Misérables ?")

❓ Question : Qui a écrit Les Misérables ?

 Réponse finale : "Les Misérables" a été écrit par Victor Hugo. C'est un roman publié en 1862, qui explore les injustices sociales et les luttes individuelles à travers l'histoire de plusieurs personnages, notamment Jean Valjean et Javert.


'"Les Misérables" a été écrit par Victor Hugo. C\'est un roman publié en 1862, qui explore les injustices sociales et les luttes individuelles à travers l\'histoire de plusieurs personnages, notamment Jean Valjean et Javert.'

**👀 Aucun outil appelé** — le LLM savait répondre tout seul avec ses connaissances. C'est le LLM qui **décide** si un outil est nécessaire.

---
## 🔗 Étape 6 — Questions qui nécessitent **plusieurs outils enchaînés**

C'est ici que ça devient vraiment impressionnant. L'agent va enchaîner les outils tout seul, dans le bon ordre.

In [ ]:
agent("Quelle température fait-il à Tokyo, et combien ça fait en Fahrenheit ? (formule : °F = °C × 9/5 + 32)")

❓ Question : Quelle température fait-il à Tokyo, et combien ça fait en Fahrenheit ? (formule : °F = °C × 9/5 + 32)

 Le LLM veut appeler : get_weather({'ville': 'Tokyo'})
 Résultat : Météo à Tokyo (Japon) : 11.5°C, vent 2.2 km/h, humidité 86%

 Le LLM veut appeler : calculator({'expression': '11.5 * 9/5 + 32'})
 Résultat : Résultat : 52.7

 Réponse finale : Il fait actuellement 11.5°C à Tokyo, ce qui correspond à environ 52.7°F.


'Il fait actuellement 11.5°C à Tokyo, ce qui correspond à environ 52.7°F.'

**👀 Regardez bien :**
1. L'agent appelle d'abord `get_weather("Tokyo")` → obtient la température en °C
2. Puis il appelle `calculator` avec la bonne formule en remplaçant la valeur
3. Enfin il formule la réponse finale

**Il a raisonné sur l'ordre des étapes tout seul.** Personne ne lui a dit "d'abord la météo, puis le calcul". C'est ça, la magie des agents.

In [ ]:
# Un exemple encore plus impressionnant
agent("Je prévois un voyage à Rome, Madrid et Berlin. Dans quelle ville fait-il le plus chaud en ce moment ? Et quelle est la différence avec la plus froide ?")

❓ Question : Je prévois un voyage à Rome, Madrid et Berlin. Dans quelle ville fait-il le plus chaud en ce moment ? Et quelle est la différence avec la plus froide ?

 Le LLM veut appeler : get_weather({'ville': 'Rome'})
 Résultat : Météo à Rome (Italie) : 15.1°C, vent 10.5 km/h, humidité 51%

 Le LLM veut appeler : get_weather({'ville': 'Madrid'})
 Résultat : Météo à Madrid (Espagne) : 19.3°C, vent 2.8 km/h, humidité 45%

 Le LLM veut appeler : get_weather({'ville': 'Berlin'})
 Résultat : Météo à Berlin (Allemagne) : 12.8°C, vent 8.2 km/h, humidité 39%

 Le LLM veut appeler : calculator({'expression': '19.3 - 12.8'})
 Résultat : Résultat : 6.5

 Réponse finale : Actuellement, Madrid est la ville la plus chaude avec une température de 19.3°C, tandis que Berlin est la plus froide à 12.8°C. La différence de température entre Madrid et Berlin est de 6.5°C.


'Actuellement, Madrid est la ville la plus chaude avec une température de 19.3°C, tandis que Berlin est la plus froide à 12.8°C. La différence de température entre Madrid et Berlin est de 6.5°C.'

**L'agent :**
1. Appelle `get_weather` **3 fois** (Rome, Madrid, Berlin)
2. Compare les températures
3. Utilise `calculator` pour la différence
4. Donne une réponse complète

**Imaginez le pouvoir** : on a codé ~50 lignes, et on a un assistant qui peut enchaîner des raisonnements complexes.

---
## Étape 7 — À vous de jouer

### Exercice 1 : posez 5 questions variées à l'agent

Testez :
- Une question où il doit utiliser `get_weather`
- Une question où il doit utiliser `calculator`
- Une question sans outil
- Une question où il doit enchaîner les 2 outils
- Une question volontairement tordue pour le piéger

In [19]:
# À vous !
q1 = agent("Quel est le météo à Hanoi?")
q2 = agent("Calcule 54*pi")
q3 = agent("When is Roland Garros")
q4 = agent("How much hotter or colder is Paris right now compared to Hanoi?")
q4 = agent("Quel est le météo à Alethkar?")

❓ Question : Quel est le météo à Hanoi?

 Le LLM veut appeler : get_weather({'ville': 'Hanoi'})
 Résultat : Météo à Hanoï (Viêt Nam) : 24.2°C, vent 5.0 km/h, humidité 80%

 Réponse finale : La météo à Hanoï est de 24,2°C avec un vent de 5,0 km/h et une humidité de 80%.
❓ Question : Calcule 54*pi

 Le LLM veut appeler : calculator({'expression': '54*3.141592653589793'})
 Résultat : Résultat : 169.64600329384882

 Réponse finale : Le calcul de \( 54 \times \pi \) est d'environ 169.65.
❓ Question : When is Roland Garros

 Réponse finale : Roland Garros, also known as the French Open, typically takes place in late May and early June. In 2024, the tournament is scheduled to be held from May 26 to June 9.
❓ Question : How much hotter or colder is Paris right now compared to Hanoi?

 Le LLM veut appeler : get_weather({'ville': 'Paris'})
 Résultat : Météo à Paris (France) : 14.7°C, vent 12.1 km/h, humidité 44%

 Le LLM veut appeler : get_weather({'ville': 'Hanoi'})
 Résultat : Météo à Hanoï (V

In [29]:
q6 = agent("What type of move is good against Dragon-type Pokemon?")

❓ Question : What type of move is good against Dragon-type Pokemon?

 Le LLM veut appeler : get_pokemon_weakness({'defending_types_str': 'Dragon'})
 Résultat : Les types d'attaque les plus efficaces contre Dragon sont Ice, Dragon, Fairy avec un multiplicateur de 2.0x.

 Réponse finale : Les types de mouvements qui sont efficaces contre les Pokémon de type Dragon sont les suivants :

- **Glace** (Ice)
- **Dragon** (Dragon)
- **Fée** (Fairy)

Ces types de mouvements infligent un multiplicateur de 2.0x de dégâts aux Pokémon Dragon.


---
 ## Étape 8 — Créer un outil pour la faiblesse et résistance des Pokémon.

In [28]:
import pandas as pd

# Load the CSV file into a DataFrame
pokemon_type_chart = pd.read_csv('/content/pokemon_type_effectiveness.csv')

# Display the first few rows and the columns to understand the structure
print('Pokémon Type Effectiveness Chart:')
display(pokemon_type_chart.head())
print('\nColumn Information:')
pokemon_type_chart.info()

Pokémon Type Effectiveness Chart:


,attacking_type,defending_type,move_effectiveness
0,Normal,Normal,1.0
1,Normal,Fire,1.0
2,Normal,Water,1.0
3,Normal,Electric,1.0
4,Normal,Grass,1.0



Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   attacking_type      324 non-null    object 
 1   defending_type      324 non-null    object 
 2   move_effectiveness  324 non-null    float64
dtypes: float64(1), object(2)
memory usage: 7.7+ KB


In [22]:
def get_pokemon_weakness(defending_types_str: str) -> str:
    """Détermine les types d'attaques les plus efficaces contre un ou deux types de Pokémon défenseurs.
    Exemple: 'Grass/Ground' ou 'Fire'."""
    defending_types = [dt.strip() for dt in defending_types_str.split('/')]

    if not 1 <= len(defending_types) <= 2:
        return "Veuillez fournir un ou deux types de Pokémon séparés par '/'. Ex: 'Grass/Ground' ou 'Fire'."

    # Validate types against the chart
    valid_types = pokemon_type_chart['defending_type'].unique()
    for d_type in defending_types:
        if d_type not in valid_types:
            return f"Le type '{d_type}' n'est pas un type de Pokémon valide dans le tableau."

    effectiveness_scores = {}

    # Get all unique attacking types from the chart
    all_attacking_types = pokemon_type_chart['attacking_type'].unique()

    for attacking_type in all_attacking_types:
        combined_effectiveness = 1.0
        for defending_type in defending_types:
            # Look up effectiveness from the DataFrame
            effectiveness_row = pokemon_type_chart[
                (pokemon_type_chart['attacking_type'] == attacking_type) &
                (pokemon_type_chart['defending_type'] == defending_type)
            ]
            if not effectiveness_row.empty:
                combined_effectiveness *= effectiveness_row['move_effectiveness'].iloc[0]
            else:
                # Should not happen if types are valid, but good for robustness
                combined_effectiveness = 0.0 # If no entry, assume no effect
                break
        effectiveness_scores[attacking_type] = combined_effectiveness

    # Filter for effective moves (score > 1.0) and sort
    effective_moves = {k: v for k, v in effectiveness_scores.items() if v > 1.0}

    if not effective_moves:
        return f"Aucun type d'attaque n'est super efficace contre {defending_types_str}."

    # Find the maximum effectiveness score
    max_effectiveness = max(effective_moves.values())

    # Get all attacking types that have this maximum effectiveness
    best_attacking_types = [
        atk_type for atk_type, score in effective_moves.items()
        if score == max_effectiveness
    ]

    if len(best_attacking_types) == 1:
        return f"Le type d'attaque le plus efficace contre {defending_types_str} est {best_attacking_types[0]} avec un multiplicateur de {max_effectiveness}x."
    else:
        return f"Les types d'attaque les plus efficaces contre {defending_types_str} sont {', '.join(best_attacking_types)} avec un multiplicateur de {max_effectiveness}x."